# LangChain Single-Agent Architecture — Lab Notes & Experiments

In [74]:
import os 
import certifi 
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.tools import tool


In [75]:
from langchain_classic.agents import create_react_agent, AgentExecutor

In [77]:
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

GEMINI_API_KEY=os.getenv('GOOGLE_API_KEY')
TAVILY_API_KEY=os.getenv('TAVILY_API_KEY')

In [78]:
search_tool = TavilySearchResults(max_results=1)

In [24]:
result = search_tool.invoke("What is capital of france")

In [79]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    google_api_key=GEMINI_API_KEY,
    temperature=1.0,  
    max_retries=2,
)

result = model.invoke("how much is 5x5 ? ")
result.text


/home/med/miniconda3/envs/langagent/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


'5 x 5 = 25'

In [ ]:
import google.generativeai as genai
genai.configure(api_key=os.environ.get("GOOGLE_API_KEY"))
for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(m.name)

In [64]:
from langsmith import Client

client = Client()
prompt = client.pull_prompt("hwchase17/react", dangerously_pull_public_prompt=True)

In [80]:
tools = [search_tool]

In [81]:
# create the agent
agent = create_react_agent(llm=model,tools=tools,prompt=prompt)

In [82]:
# executor
agent_executor = AgentExecutor(agent=agent,
                               tools=tools,
                               verbose=True
                               )

agent_executor


AgentExecutor(verbose=True, agent=RunnableAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_log_to_str(x['intermediate_steps']))
})
| PromptTemplate(input_variables=['agent_scratchpad', 'input'], input_types={}, partial_variables={'tools': 'tavily_search_results_json - A search engine optimized for comprehensive, accurate, and trusted results. Useful for when you need to answer questions about current events. Input should be a search query.', 'tool_names': 'tavily_search_results_json'}, metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: 

In [83]:
# running the executor
reponse = agent_executor.invoke({"input":("find the capital of morcco"
                                          "and then finds its current weather.")})



> Entering new AgentExecutor chain...


/home/med/miniconda3/envs/langagent/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Thought:
Action: tavily_search_results_json
Action Input: capital of Morocco[{'title': 'Rabat - Wikipedia', 'url': 'https://en.wikipedia.org/wiki/Rabat', 'content': 'Rabat is the capital city of Morocco, located on the northwestern coasts of the Atlantic Ocean. Rabat is the country\'s seventh-largest city with a population of 515,619 (2024) and the second largest metropolitan area after Casablanca, with a population of approximately 2.1 million (2025). It is also the capital city of the Rabat-Salé-Kénitra administrative region. Rabat is located on the Atlantic Ocean at the mouth of the river Bou Regreg, opposite Salé, the city\'s main commuter town. [...] Edit links\n\n Article\n Talk\n\n Read\n Edit\n View history\n\nTools\n\nActions\n\n Read\n Edit\n View history\n\nGeneral\n\n What links here\n Related changes\n Upload file\n Permanent link\n Page information\n Cite this page\n Get shortened URL\n Switch to legacy parser\n\nPrint/export\n\n Download as PDF\n Printable version\n\nIn 

/home/med/miniconda3/envs/langagent/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Action: tavily_search_results_json
Action Input: current weather in Rabat Morocco[{'title': 'Rabat, Rabat-Salé-Kénitra, Morocco Monthly Weather', 'url': 'https://www.accuweather.com/en/ma/rabat/245072/october-weather/245072', 'content': "Rabat's October 2026 forecast shows daily high temperatures ranging from 73° to 77°, with overnight lows between 47° and 59°. The average high for October is 75", 'score': 0.93191636}]

/home/med/miniconda3/envs/langagent/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Action: tavily_search_results_json
Action Input: current weather in Rabat today[{'title': 'Weather for Rabat, Morocco', 'url': 'https://www.timeanddate.com/weather/morocco/rabat', 'content': '72 °F Clear. Feels Like: 74 °F Forecast: 76 / 54 °F Wind: 6 mph ↑ from Northwest Location: Rabat-Sale Current Time: Oct 25, 2026 at 1:11:13 pm … 18 "Hg', 'score': 0.9353998}]

/home/med/miniconda3/envs/langagent/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Final Answer: The capital of Morocco is Rabat. The current weather in Rabat is clear with a temperature of around 72°F (feels like 74°F) and winds from the northwest at 6 mph.

> Finished chain.


In [84]:
print(reponse["output"])

The capital of Morocco is Rabat. The current weather in Rabat is clear with a temperature of around 72°F (feels like 74°F) and winds from the northwest at 6 mph.
